# 🔄 The Pivot — GRPO Training
**Meta PyTorch OpenEnv Hackathon 2026**

Just run every cell top to bottom. Runtime → Change runtime type → **T4 GPU** first.

In [ ]:
# Cell 1 — Check GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — go to Runtime → Change runtime type → T4')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Cell 2 — Install packages
%%capture
!pip install -q openenv-core "transformers>=4.45.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" wandb pydantic numpy python-dotenv
print('done')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Saving to: {SAVE_DIR}')

In [ ]:
# Cell 4 — Clone project
import os, sys
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler
sys.path.insert(0, '/content/meta_scaler')
os.chdir('/content/meta_scaler')

try:
    from models import PivotAction, ActionType, PivotObservation
    from server.pivot_environment import ThePivotEnvironment
    from server.prompt_encoder import encode_to_messages
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports OK!')
except ImportError as e:
    print(f'❌ Import error: {e}')

In [ ]:
# Cell 5 — W&B login
import wandb
wandb.login()   # paste your key from wandb.ai/settings
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

In [ ]:
# Cell 6 — Load model (Qwen2.5-1.5B + LoRA)
# Using 1.5B instead of 0.5B: 3x more parameters → better startup reasoning.
# 1.5B at 4-bit ≈ 2GB VRAM for weights + ref copy ≈ 5GB total → fits T4 (15GB).
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME     = 'Qwen/Qwen2.5-1.5B-Instruct'   # ← upgraded from 0.5B
DEVICE         = 'cuda'
MAX_SEQ_LEN    = 512
MAX_NEW_TOKENS = 8

print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map='auto',
    trust_remote_code=True,
)

model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
))

# Allows gradients to flow into LoRA adapters through frozen 4-bit base layers
model.enable_input_require_grads()

model.print_trainable_parameters()
gc.collect(); torch.cuda.empty_cache()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model ready!  GPU: {used:.2f} GB used / {total:.1f} GB total  ({total-used:.1f} GB free)')

In [ ]:
# Cell 7 — Helpers
import re, random, gc, numpy as np
from models import PivotAction, ActionType, PivotObservation
from server.pivot_environment import ThePivotEnvironment
from server.prompt_encoder import encode_to_messages

ACTION_MAP = {
    'execute':   ActionType.EXECUTE,   'pivot':     ActionType.PIVOT,
    'research':  ActionType.RESEARCH,  'fundraise': ActionType.FUNDRAISE,
    'hire':      ActionType.HIRE,      'cut_costs': ActionType.CUT_COSTS,
    'cut':       ActionType.CUT_COSTS,
}

# Full list for parsing model output (PIVOT allowed here — model can decide to pivot)
ACTION_LIST = [ActionType.EXECUTE, ActionType.PIVOT, ActionType.RESEARCH,
               ActionType.FUNDRAISE, ActionType.HIRE, ActionType.CUT_COSTS]

# Random exploration pool — PIVOT intentionally excluded.
# Reason: PIVOT costs -3mo runway each time. 6 random actions with 1/6 PIVOT chance
# = ~6 random PIVOTs per episode at ε=0.6 → company always dies.
# PIVOT should only happen when the MODEL explicitly decides it's time.
# Random exploration uses safe actions: EXECUTE, RESEARCH, HIRE, FUNDRAISE, CUT_COSTS.
RANDOM_ACTION_LIST = [ActionType.EXECUTE, ActionType.RESEARCH,
                      ActionType.FUNDRAISE, ActionType.HIRE, ActionType.CUT_COSTS]


def _parse(text: str) -> ActionType:
    w = re.sub(r'[^a-z_]', '', text.lower().split()[0]) if text.strip() else 'execute'
    return ACTION_MAP.get(w, ActionType.EXECUTE)


@torch.no_grad()
def generate_action(obs):
    text = tokenizer.apply_chat_template(encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
    inp  = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    out  = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                          do_sample=True, temperature=0.9, top_p=0.9,
                          pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    del inp, out
    return decoded, _parse(decoded)


def get_log_prob(prompt: str, completion: str) -> torch.Tensor:
    """Tokenize completion first, truncate prompt from left so completion always survives."""
    comp_ids = tokenizer(completion, return_tensors='pt',
                         add_special_tokens=False)['input_ids'][0]
    if comp_ids.shape[0] == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)

    prompt_ids = tokenizer(prompt, return_tensors='pt',
                           truncation=True,
                           max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]

    full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
    p_len    = prompt_ids.shape[0]

    out         = model(input_ids=full_ids)
    comp_logits = out.logits[0, p_len - 1:-1, :].clone()
    target_ids  = full_ids[0, p_len:].clone()
    del out, full_ids
    torch.cuda.empty_cache()

    lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
    return lp[range(len(target_ids)), target_ids].mean()


def run_episode(scenario: dict, seed: int = 0, epsilon: float = 0.3) -> list[dict]:
    """
    Runs one episode. Key design choices:
    - Unique seed → different environment each episode
    - ε-greedy: random actions come from RANDOM_ACTION_LIST (no PIVOT)
    - PIVOT only comes from the model's own output → fewer catastrophic early pivots
    """
    env = ThePivotEnvironment(scenario=scenario, rng_seed=seed)
    obs = env.reset()
    steps = []
    for _ in range(60):
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
        if random.random() < epsilon:
            # Random exploration: safe actions only — no PIVOT
            action_type = random.choice(RANDOM_ACTION_LIST)
            completion  = action_type.value.upper()
        else:
            completion, action_type = generate_action(obs)
        next_obs = env.step(PivotAction(action_type=action_type))
        steps.append({'prompt': prompt, 'completion': completion,
                      'action': action_type.value, 'reward': float(next_obs.reward or 0),
                      'step': next_obs.step, 'runway': next_obs.runway_remaining})
        obs = next_obs
        if obs.done: break
    return steps


# ── Sanity check ──────────────────────────────────────────────────────────────
model.train()
env  = ThePivotEnvironment()
obs  = env.reset()
_prompt = tokenizer.apply_chat_template(encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
_lp = get_log_prob(_prompt, 'EXECUTE')
print(f'Gradient check:')
print(f'  log_prob     = {_lp.item():.4f}')
print(f'  requires_grad= {_lp.requires_grad}   ← must be True')
print(f'  grad_fn      = {_lp.grad_fn}   ← must NOT be None')
assert _lp.requires_grad and _lp.grad_fn is not None, '❌ Gradients not flowing!'
model.eval()
print('✅ Helpers ready!  PIVOT removed from random pool — model must earn it.')

In [ ]:
# Cell 8 — Config, optimizer, curriculum, GRPO update (with KL penalty + advantage clipping)
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
import copy

CONFIG = {
    'n_episodes':        150,
    'lr':                3e-5,   # slightly lower for 1.5B — more stable
    'grad_clip':         1.0,
    'grpo_steps_sample': 8,      # more steps per update (was 6) — better signal
    'save_every':        25,
    'log_every':         5,
    'kl_beta':           0.04,
    'adv_clip':          2.0,    # NEW: clip advantages to [-2, +2] → stops loss oscillation
}
optimizer  = AdamW(model.parameters(), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)

# Reference model for KL penalty — frozen copy of the initial policy
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)
print('Reference model frozen for KL penalty.')


def get_log_prob_nograd(prompt: str, completion: str) -> float:
    with torch.no_grad():
        comp_ids = tokenizer(completion, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        if comp_ids.shape[0] == 0:
            return 0.0
        prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True,
                               max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]
        full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
        p_len = prompt_ids.shape[0]
        out = ref_model(input_ids=full_ids)
        comp_logits = out.logits[0, p_len - 1:-1, :]
        target_ids = full_ids[0, p_len:]
        lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
        val = lp[range(len(target_ids)), target_ids].mean().item()
        del out, full_ids; torch.cuda.empty_cache()
        return val


def grpo_update(episode_steps: list[dict]) -> dict:
    """
    GRPO with KL penalty + advantage clipping.
    - Advantage clipping to [-2, +2]: stops a single extreme step from
      flipping all gradients, which caused the wild loss oscillation (3.05 → -2.45).
    - KL penalty: keeps policy close to reference, prevents reward hacking.
    """
    model.train()
    optimizer.zero_grad()

    rewards = np.array([s['reward'] for s in episode_steps], dtype=np.float32)
    # Normalize advantages within episode
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6) if rewards.std() > 1e-6 else rewards - rewards.mean()
    # CLIP: no single step should dominate with advantage > 2σ
    adv = np.clip(adv, -CONFIG['adv_clip'], CONFIG['adv_clip'])

    sample_k = min(CONFIG['grpo_steps_sample'], len(episode_steps))
    indices  = random.sample(range(len(episode_steps)), sample_k)

    total_loss = 0.0
    total_kl   = 0.0
    did_bwd    = False

    for idx in indices:
        sd = episode_steps[idx]
        lp  = get_log_prob(sd['prompt'], sd['completion'])
        ref = get_log_prob_nograd(sd['prompt'], sd['completion'])

        grpo_loss = (-lp * float(adv[idx])) / sample_k
        kl        = (lp - ref) / sample_k
        kl_loss   = CONFIG['kl_beta'] * kl
        loss      = grpo_loss + kl_loss

        if loss.requires_grad and loss.grad_fn is not None:
            loss.backward()
            did_bwd = True

        total_loss += grpo_loss.item()
        total_kl   += kl.item()
        del lp, kl, loss
        gc.collect(); torch.cuda.empty_cache()

    if did_bwd:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()
    optimizer.zero_grad()
    model.eval()
    gc.collect(); torch.cuda.empty_cache()
    return {'loss': total_loss, 'kl': total_kl / sample_k}


print('✅ Config ready:', CONFIG)

In [ ]:
# Cell 9 — TRAIN
# ─────────────────────────────────────────────────────────────────────────────
# ROOT CAUSE OF PREVIOUS BUG:
#   run_episode(scenario) was called with NO seed → every episode used seed=0
#   → identical environment every time → same reward (-158.0) locked forever.
#   Also: epsilon was never passed/decayed → exploration stayed flat.
#
# FIXES in this cell:
#   1. seed = ep * 13 + 7     → each episode gets a DIFFERENT deterministic env
#   2. epsilon starts at 0.60 → high early exploration (model is untrained)
#   3. epsilon decays × 0.97  → by ep 50 ≈ 0.18, by ep 100 ≈ 0.05
# ─────────────────────────────────────────────────────────────────────────────
import os
os.environ.pop('WANDB_MODE', None)

run = wandb.init(project=WANDB_PROJECT, name='grpo-qwen0.5b-pivot',
                 config=CONFIG, tags=['grpo','qwen0.5b','pivot','hackathon'])
print(f'W&B: {run.url}')
print(f'Starting tier 1/5: {DIFFICULTY_LADDER[0]}')
print('─' * 72)

# ── Epsilon schedule ──────────────────────────────────────────────────────────
epsilon   = 0.60   # start high — model is untrained, need diverse actions
EPS_DECAY = 0.97   # ×0.97 each ep → ep50 ≈ 0.18, ep100 ≈ 0.05
EPS_MIN   = 0.08   # floor — keep 8% random throughout to avoid collapse

episode_rewards, episode_lengths, all_losses = [], [], []
best_reward = float('-inf')
model.eval()

for ep in range(1, CONFIG['n_episodes'] + 1):
    scenario      = curriculum.sample_scenario()
    scenario_name = scenario.get('name', 'default')

    # ── FIX 1: unique seed per episode ───────────────────────────────────────
    seed = ep * 13 + 7   # deterministic but different every episode

    # ── FIX 2: pass seed + epsilon (were missing before → seed=0 always) ─────
    steps = run_episode(scenario, seed=seed, epsilon=epsilon)

    ep_reward   = sum(s['reward'] for s in steps)
    ep_length   = len(steps)
    survived    = ep_length >= 60
    pivot_count = sum(1 for s in steps if s['action'] == 'PIVOT')
    unique_acts = len(set(s['action'] for s in steps))

    episode_rewards.append(ep_reward)
    episode_lengths.append(ep_length)
    if ep_reward > best_reward:
        best_reward = ep_reward

    loss_stats = grpo_update(steps)
    all_losses.append(loss_stats['loss'])
    curriculum.record_result(ep_reward, survived)

    # ── FIX 3: decay epsilon each episode ────────────────────────────────────
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    gpu_gb = torch.cuda.memory_allocated() / 1e9
    mean20 = float(np.mean(episode_rewards[-20:]))
    print(
        f'Ep {ep:3d}/{CONFIG["n_episodes"]} | {scenario_name:16s} | '
        f'T{curriculum.current_tier+1} | steps {ep_length:3d} | '
        f'reward {ep_reward:+7.1f} | best {best_reward:+7.1f} | '
        f'loss {loss_stats["loss"]:7.5f} | ε={epsilon:.2f} | '
        f'pivots {pivot_count} | acts {unique_acts} | '
        f'{"SURVIVED" if survived else "died"} | GPU {gpu_gb:.1f}GB'
    )

    if ep % CONFIG['log_every'] == 0:
        wandb.log({
            'episode':          ep,
            'ep_reward':        ep_reward,
            'best_reward':      best_reward,
            'mean_reward_20ep': mean20,
            'ep_length':        ep_length,
            'survived':         int(survived),
            'pivot_count':      pivot_count,
            'unique_actions':   unique_acts,
            'loss':             loss_stats['loss'],
            'kl':               loss_stats.get('kl', 0),
            'epsilon':          epsilon,
            'curriculum_tier':  curriculum.current_tier + 1,
            'gpu_gb':           gpu_gb,
            'scenario':         scenario_name,
        })

    if curriculum.should_advance() and curriculum.advance_tier():
        new = DIFFICULTY_LADDER[curriculum.current_tier]
        print(f'\n🎓 CURRICULUM ADVANCE → Tier {curriculum.current_tier+1}/5: {new}\n')
        wandb.log({'curriculum_advance': curriculum.current_tier, 'episode': ep})

    if ep % CONFIG['save_every'] == 0:
        ckpt = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        print(f'💾 Checkpoint → {ckpt}')

print('\n' + '='*72)
print('✅ TRAINING COMPLETE!')
print(f'Mean reward last 20 ep : {np.mean(episode_rewards[-20:]):.1f}')
print(f'Best reward            : {best_reward:.1f}')
print(f'Survival rate          : {sum(l>=60 for l in episode_lengths)/len(episode_lengths):.0%}')
print(f'Final tier reached     : {curriculum.current_tier+1}/5')
print('='*72)

In [ ]:
# Cell 10 — Save model + close W&B
FINAL = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
print(f'✅ Model saved to {FINAL}')
wandb.finish()
print('✅ W&B closed')

In [ ]:
# Cell 12 — Save training plots as PNG (judges need these committed to the repo)
import matplotlib.pyplot as plt
import numpy as np, os, json, pathlib

PLOTS_DIR = pathlib.Path('/content/meta_scaler/docs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Also save to Drive for safekeeping
DRIVE_PLOTS = pathlib.Path(f'{SAVE_DIR}/plots')
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

def _smooth(xs, k=10):
    if len(xs) < k: return xs
    return np.convolve(xs, np.ones(k)/k, mode='valid')

# ── Reward curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_rewards, alpha=0.3, color='tab:blue', label='per-episode')
if len(episode_rewards) >= 10:
    ax.plot(range(9, len(episode_rewards)), _smooth(episode_rewards, 10),
            color='tab:blue', linewidth=2, label='10-ep moving avg')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Episode'); ax.set_ylabel('Total reward')
ax.set_title('The Pivot — GRPO training reward curve (Qwen2.5-0.5B)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'reward_curve.png', DRIVE_PLOTS / 'reward_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Loss curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(all_losses, alpha=0.3, color='tab:red', label='per-episode loss')
if len(all_losses) >= 10:
    ax.plot(range(9, len(all_losses)), _smooth(all_losses, 10),
            color='tab:red', linewidth=2, label='10-ep moving avg')
ax.set_xlabel('Episode'); ax.set_ylabel('GRPO loss')
ax.set_title('The Pivot — GRPO training loss')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'loss_curve.png', DRIVE_PLOTS / 'loss_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Survival / episode length ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_lengths, color='tab:green', alpha=0.6)
ax.axhline(60, color='gray', linestyle='--', label='max (60)')
ax.set_xlabel('Episode'); ax.set_ylabel('Steps survived')
ax.set_title('The Pivot — episode length over training')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'survival_curve.png', DRIVE_PLOTS / 'survival_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Dump raw metrics as JSON for the README ───────────────────────────
metrics = {
    'n_episodes': len(episode_rewards),
    'mean_reward_last_20': float(np.mean(episode_rewards[-20:])),
    'best_reward':         float(max(episode_rewards)),
    'survival_rate':       float(sum(l>=60 for l in episode_lengths) / len(episode_lengths)),
    'final_tier':          curriculum.current_tier + 1,
}
for p in [PLOTS_DIR / 'metrics.json', DRIVE_PLOTS / 'metrics.json']:
    with open(p, 'w') as f: json.dump(metrics, f, indent=2)
print('\n✅ All plots + metrics saved. Metrics:', metrics)
print('Now git push these:')
print('  cd /content/meta_scaler && git add docs/plots && git commit -m "Add training plots" && git push')

In [ ]:
# Cell 11 — Evaluate trained vs baselines
from training.baseline_agent import RandomAgent, StubbornAgent, PanicAgent, run_episodes
import json

class TrainedAgent:
    name = 'trained_llm'
    def act(self, obs):
        _, at = generate_action(obs)
        return PivotAction(action_type=at)

c = AdaptiveCurriculum()
agents = [RandomAgent(), StubbornAgent(), PanicAgent(), TrainedAgent()]
print(f'{"Scenario":22s} | {"Agent":12s} | Reward   | Survival | Pivots')
print('-' * 66)
results = []
for name in DIFFICULTY_LADDER:
    sc = c._all_scenarios.get(name)
    if not sc: continue
    for agent in agents:
        r = run_episodes(agent, sc, 20)
        results.append(r)
        print(f'{name:22s} | {agent.name:12s} | {r["mean_reward"]:+7.1f} | {r["survival_rate"]:6.0%}   | {r["pivot_rate"]:6.0%}')

with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\n✅ Evaluation done — saved to Drive')